# Data Preprocessing

Flatten collected YouTube comments and replies into clean rows for downstream analysis.


In [1]:
import json
from pathlib import Path
import pandas as pd
import random 
from langdetect import detect, LangDetectException
from collections import Counter
import nltk
import string
import re
import html
import unicodedata
import emoji
from nltk.corpus import stopwords


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
VIDEO_DATA_PATH = DATA_DIR / "video_data.json"
PROCESSED_VIDEO_DATA_PATH = DATA_DIR / "video_data_processed.json"
MET_GALA_ENTITIES_PATH = DATA_DIR / "met_gala_entities.json"

RANDOM_SEED = 42

nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)


True

In [2]:
with open(VIDEO_DATA_PATH, "r", encoding="utf-8") as f:
    video_data = json.load(f)["videos"]

print("Video data loaded")
print("Total videos:", len(video_data))
print("Total collected comment rows:", sum(len(video.get("comments", [])) for video in video_data))


Video data loaded
Total videos: 110
Total collected comment rows: 63250


In [3]:
comments_flattened = []
seen_comment_ids = set()
duplicate_comment_rows = 0
for video in video_data:
    video_context = {
        "video_id": video.get("videoId"),
        "video_title": video.get("title"),
        "channel_id": video.get("channelId"),
        "channel_title": video.get("channelTitle"),
        "video_published_at": video.get("publishedAt"),
        "video_view_count": video.get("viewCount", 0),
        "video_like_count": video.get("likeCount", 0),
        "video_available_comment_count": video.get("commentCount", 0),
    }

    for comment in video.get("comments", []):
        comment_id = comment.get("commentId")
        if comment_id in seen_comment_ids:
            duplicate_comment_rows += 1
            continue
        seen_comment_ids.add(comment_id)

        comments_flattened.append({
            **video_context,
            "comment_id": comment_id,
            "comment_text": comment.get("text", ""),
            "comment_author_id": comment.get("authorId"),
            "comment_author": comment.get("author"),
            "comment_published_at": comment.get("publishedAt"),
            "comment_updated_at": comment.get("updatedAt"),
            "comment_like_count": comment.get("likeCount", 0),
            "is_reply": comment.get("isReply", False),
            "parent_comment_id": comment.get("parentCommentId"),
            "reply_to_author_id": comment.get("replyToAuthorId"),
            "top_level_reply_count": comment.get("totalReplyCount", 0),
            "text_length": len(comment.get("text", "") or ""),
        })


In [27]:

total_comments = len(comments_flattened)
total_replies = sum(1 for c in comments_flattened if c.get("is_reply") == True)
total_parent_comments = total_comments - total_replies
total_empty_comments = sum(1 for c in comments_flattened if not c.get("comment_text", "").strip())

print("COMMENT FLATTENING SUMMARY")
print("=" * 80)
print(f"Flattened comment rows: {total_comments}")
print(f"Duplicate comment rows removed: {duplicate_comment_rows}\n")

print(f"Total comments: {total_comments}")
print(f"Total parent comments: {total_parent_comments}")
print(f"Total replies: {total_replies}")
print(f"Total empty comments: {total_empty_comments}")

COMMENT FLATTENING SUMMARY
Flattened comment rows: 46829
Duplicate comment rows removed: 202

Total comments: 46829
Total parent comments: 36091
Total replies: 10738
Total empty comments: 13


In [5]:

TOTAL_RANDOM_SAMPLES = 25

print("\nRANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES")
print("=" * 80)
random.seed(RANDOM_SEED)
random_sample_indices = random.sample(range(len(comments_flattened)), min(TOTAL_RANDOM_SAMPLES, len(comments_flattened)))
for i, idx in enumerate(random_sample_indices):
    text = comments_flattened[idx]['comment_text'].strip().replace('\n', ' ').replace('\r', '')
    text = ' '.join(text.split())
    print(f"[{i+1}/{TOTAL_RANDOM_SAMPLES}] {text[:150]:<10}")



RANDOM SAMPLE OF COMMENTS TO IDENTIFY ISSUES
[1/25] there is the stereotype and then there is bad. u won the award for both. just to show u just cause u got monie, doesnt mean shite. especially abiut fa
[2/25] 14:30 in my opinion, it needed a big sumptuous cloak, maybe with a hood and gloves, to be more evocative of how luscious and involved klimt’s pieces a
[3/25] what is cara doing :/
[4/25] I’m actually crying laughing on my way to work! Love you Zach! Can’t wait to see you hosting one day ❤
[5/25] Nicole Kidman is always pure class. She is stunning.
[6/25] Megyn is sooo jealous she wasn't invited
[7/25] My only look at the met gala, thanks Garrron! Getting Halloween in springtime vibes 🤡👻👽👀
[8/25] I agree with so many of your critiques. The fact Anna didn't even bother with the theme and wore a variation of a previous dress let's me know how fri
[9/25] ​@prpowell4038😢 for the fact she's entitled she might be acting the same way as North. Get out of gay Zee's and be-yawn🥱nce's asse

In [6]:
def detect_language(text):
    """Detect language, returning ISO code."""
    try:
        if not text or not len(text.strip()):
            return 'en'
        return detect(text)
    except LangDetectException:
        return 'unknown'

# Collect comment rows from the rows list
language_results = [(comment, detect_language(comment.get('comment_text', ''))) for comment in comments_flattened]
language_counter = Counter(lang for _, lang in language_results)

In [7]:
TOP_LANGUAGE_COUNT = 10
TOTAL_NON_ENGLISH_EXAMPLES = 20

total_comments = len(language_results)

print(f"\nTOP {TOP_LANGUAGE_COUNT} LANGUAGE DETECTIONS")
print("=" * 80)
for i, (lang, count) in enumerate(language_counter.most_common(TOP_LANGUAGE_COUNT), start=1):
    pct = 100 * count / total_comments
    print(f"[{i}] {lang} {count:,} ({pct:.2f}%)")
    if i == 10:
        break


TOP 10 LANGUAGE DETECTIONS
[1] en 46,829 (74.28%)
[2] unknown 1,679 (2.66%)
[3] so 1,563 (2.48%)
[4] pt 1,144 (1.81%)
[5] de 1,074 (1.70%)
[6] tl 1,072 (1.70%)
[7] af 939 (1.49%)
[8] fr 831 (1.32%)
[9] et 811 (1.29%)
[10] id 748 (1.19%)


In [8]:
COMMENT_TRUNCATION_LENGTH = 200
ENGLISH_FILTER = "en"

# Extract full comment data for English comments
english_comments = [comment for comment, lang in language_results if lang == ENGLISH_FILTER]
for comment in english_comments:
    comment["language"] = ENGLISH_FILTER

# Overwrite comments flattened with English filtered list
comments_flattened = english_comments

# Extract truncated comment data for non-English comments for test display
non_english_comments = [comment['comment_text'][:COMMENT_TRUNCATION_LENGTH] for comment, lang in language_results if lang != 'en']

random.seed(RANDOM_SEED)
random_non_english = random.sample(non_english_comments, min(TOTAL_NON_ENGLISH_EXAMPLES, len(non_english_comments)))
random_english = random.sample(comments_flattened, min(TOTAL_NON_ENGLISH_EXAMPLES, len(comments_flattened)))

In [9]:

print("\nRANDOM NON-ENGLISH COMMENTS REMOVED:")
print("=" * 80)
for idx, text in enumerate(random_non_english):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {text}")



RANDOM NON-ENGLISH COMMENTS REMOVED:
[1/20] All jokers😂
[2/20] Rose amm ammm aaa and😅😂
[3/20] KUN！❤❤❤
[4/20] Muy Lindo ❤❤❤❤❤
[5/20] Depends
[6/20] "Cardeeb" 😂😂😂
[7/20] Real Life Hunger Games
[8/20] ALYSSAAAAAAAAAAAAAAAA😮😮😮😮😮😮😮❤❤❤OMGGGG
[9/20] ❤classic❤
[10/20] Vine por las Pinks. Que emocionnnn❤❤❤❤❤
[11/20] Beautiful 💯 queen 👸 Selena ❤
[12/20] Ouaou Jaafar mais quesqu il est mignon normal c est un Jackson
[13/20] יהושע י"ז 17
ויאמר יהושע לשבטי יוסף, לאפרים ולמנשה: "רבים אתם וחזקים מאוד. לא רק חלקה אחת תהיה לכם."Μάρκος 1:13
και έμεινε στην έρημο σαράντα ημέρες, πειραζόμενος από τον Σατανά. Ήταν μαζί με τα άγρια
[14/20] doja cat dressed in ductape
[15/20] lisa littely ate
[16/20] yes!!!!
[17/20] @13:47 DIABOLICAL
[18/20] Léna ?
[19/20] ဘာဝတ်စုံ‌တွေမှန်းကို မသိဘူး မကြိုက်ဘူး
[20/20] Love u Jisoooooooo


In [10]:
print("\nRANDOM ENGLISH COMMENTS REMAINING:")
print("=" * 80)
for idx, comment in enumerate(random_english):
    print(f"[{idx+1}/{TOTAL_NON_ENGLISH_EXAMPLES}] {comment['comment_text'][:COMMENT_TRUNCATION_LENGTH]}")



RANDOM ENGLISH COMMENTS REMAINING:
[1/20] Gigi Hadid why even bother to put a dress on😂😂
[2/20] Tyla was a tragedy.. what happened😢
[3/20] I love he’s so honest, but I think she did a great job because he doesn’t really seem to be feeling it
[4/20] They’re obviously quite new to this with all the posing. There’s no natural flare 😂
[5/20] Thank you for this 🙏🏼🙏🏼 every year plz
[6/20] If anyone is wondering, the marble look was inspired by the sculpture "The veiled virgin".
[7/20] Lena looks tarred and feathered..bed bug bites, and oh those tattoos! Stunning
[8/20] There’s very little natural about these bodies anymore. This strange perversion of what used to be beautiful
[9/20] Lisa is the fashion icon of kpop✨️ 😍
[10/20] Its disrespectful.  That wasn't cute.
[11/20] Do you have any recommendations on how to get fashion education books (im envious of that library behind you)…
[12/20] My thoughts EXACTLY.
[13/20] At 14.09 she’s doing the Rachel zegler meme !! I may have spelt her last n

In [11]:
english_count = len(english_comments)
removed_count = total_comments - english_count
english_pct = 100 * english_count / total_comments
removed_pct = 100 * removed_count / total_comments

print("ENGLISH FILTERING SUMMARY")
print(80 * "=")
print(f"\nEnglish kept: {english_count} ({english_pct:.1f}%)")
print(f"Non-English removed: {removed_count} ({removed_pct:.1f}%)")

ENGLISH FILTERING SUMMARY

English kept: 46829 (74.3%)
Non-English removed: 16219 (25.7%)


In [12]:
# Preprocessing constants and helper functions
URL_PATTERN = re.compile(r'https?://\S+')
TIMESTAMP_PATTERN = re.compile(r'\b\d{1,2}:\d{2}(?::\d{2})?\b')
MENTION_PATTERN = re.compile(r'@[\w.-]+[\w]')
DIGIT_PATTERN = re.compile(r'\d+')
PUNCT_PATTERN = re.compile(r'[^\w\s]')
ENTITY_SEPARATOR_PATTERN = re.compile(r'[^a-z\s]+')

TWEET_TOKENISER = nltk.tokenize.TweetTokenizer(
    reduce_len=True,
    strip_handles=True,
    preserve_case=False,
)

PUNCTUATION = list(string.punctuation)
TWEET_STEMMER = nltk.stem.PorterStemmer()
STOP_WORDS = set(stopwords.words('english')) | set(PUNCTUATION)

def remove_html_entities(text):
    return html.unescape(text or "")

def remove_urls(text):
    return URL_PATTERN.sub("", text)

def remove_timestamps(text):
    return TIMESTAMP_PATTERN.sub("", text)

def remove_mentions(text):
    return MENTION_PATTERN.sub("", text)

def remove_accents(text):
    return unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()

def remove_unicode(text):
    return text.encode("ascii", "ignore").decode()

def remove_digits(text):
    return DIGIT_PATTERN.sub(" ", text)

def strip_punctuation(text):
    return PUNCT_PATTERN.sub(" ", text)

def strip_entity_separators(text):
    return ENTITY_SEPARATOR_PATTERN.sub(" ", text)

def remove_emojis(text):
    return emoji.replace_emoji(text, replace="")

def normalise_whitespace(text):
    return " ".join(text.split())

def tokenize(text):
    return TWEET_TOKENISER.tokenize(text)

def remove_stopwords(tokens):
    return [token for token in tokens if token not in STOP_WORDS]

def stem_tokens(tokens):
    return [TWEET_STEMMER.stem(token) for token in tokens]

def clean_common_text(text):
    text = remove_html_entities(text)
    text = remove_urls(text)
    text = remove_timestamps(text)
    return remove_mentions(text)

def clean_for_entity_matching(text):
    text = clean_common_text(text)
    text = remove_accents(text.lower())
    text = strip_entity_separators(text)
    return normalise_whitespace(text)

def sentiment_analysis_clean(text):
    """Minimal cleaning appropriate for VADER and BERT sentiment analysis."""
    return normalise_whitespace(clean_common_text(text))

def clean_for_topic_text(text):
    text = clean_for_entity_matching(text)
    text = remove_unicode(text)
    text = remove_digits(text)
    text = strip_punctuation(text)
    return normalise_whitespace(text)

def clean_for_topic_tokens(text):
    text = clean_for_topic_text(text)
    tokens = tokenize(text)
    return remove_stopwords(tokens)

def clean_for_topic_stemmed_tokens(text):
    return stem_tokens(clean_for_topic_tokens(text))

def purify_text(text, show_changes=False):
    """Return heavily cleaned, tokenized text with stopwords removed."""
    if not show_changes:
        return clean_for_topic_tokens(text)

    history = {}
    text = remove_html_entities(text)
    history["remove_html_entities"] = text
    text = remove_urls(text)
    history["remove_urls"] = text
    text = remove_timestamps(text)
    history["remove_timestamps"] = text
    text = remove_mentions(text)
    history["remove_mentions"] = text
    text = text.lower().strip()
    history["lowercase_and_strip"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace"] = text
    text = remove_unicode(text)
    history["remove_unicode"] = text
    text = remove_digits(text)
    history["remove_digits"] = text
    text = strip_punctuation(text)
    history["strip_punctuation"] = text
    text = normalise_whitespace(text)
    history["normalise_whitespace_after_punctuation"] = text
    tokens = tokenize(text)
    history["tokenize"] = tokens
    tokens = remove_stopwords(tokens)
    history["remove_stopwords"] = tokens
    history["stem_tokens"] = stem_tokens(tokens)
    return history


In [13]:
# Store lightly cleaned text for exact entity matching
for comment in comments_flattened:
    comment["comment_text_entity"] = clean_for_entity_matching(comment.get("comment_text", ""))

print("Lightly cleaned text ready for entity matching")


Lightly cleaned text ready for entity matching


In [14]:
unique_videos = set(comment['video_id'] for comment in comments_flattened)
unique_channels = set(comment['channel_id'] for comment in comments_flattened)
unique_authors = set(comment['comment_author'] for comment in comments_flattened)

SHORT_COMMENT_LENGTH = 8
short_comments = [comment for comment in comments_flattened if len(comment['comment_text']) < SHORT_COMMENT_LENGTH]
comment_lengths = [len(comment['comment_text']) for comment in comments_flattened]

comments_with_urls = [c for c in comments_flattened if URL_PATTERN.search(c['comment_text'])]
comments_with_timestamps = [c for c in comments_flattened if TIMESTAMP_PATTERN.search(c['comment_text'])]
comments_with_mentions = [c for c in comments_flattened if MENTION_PATTERN.search(c['comment_text'])]
comments_with_digits = [c for c in comments_flattened if DIGIT_PATTERN.search(c['comment_text'])]
comments_with_punctuation = [c for c in comments_flattened if PUNCT_PATTERN.search(c['comment_text'])]

comments_with_urls_pct = 100 * len(comments_with_urls) / len(comments_flattened)
comments_with_timestamps_pct = 100 * len(comments_with_timestamps) / len(comments_flattened)
comments_with_mentions_pct = 100 * len(comments_with_mentions) / len(comments_flattened)
comments_with_digits_pct = 100 * len(comments_with_digits) / len(comments_flattened)
comments_with_punctuation_pct = 100 * len(comments_with_punctuation) / len(comments_flattened)

# For author, video, and channel distributions
video_counter = Counter(comment['video_id'] for comment in comments_flattened)
channel_counter = Counter(comment['channel_id'] for comment in comments_flattened)
author_counter = Counter(comment['comment_author'] for comment in comments_flattened)

# Print all details at bottom
print("BASIC DATA EXPLORATION")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Unique videos: {len(unique_videos)}")
print(f"Unique channels: {len(unique_channels)}")
print(f"Unique authors: {len(unique_authors)}\n")

print(f"Short comments (<{SHORT_COMMENT_LENGTH} chars): {len(short_comments)}")
print(f"Comments w/ URLs: {len(comments_with_urls)} ({comments_with_urls_pct:.2f}%)")
print(f"Comments w/ timestamps: {len(comments_with_timestamps)} ({comments_with_timestamps_pct:.2f}%)")
print(f"Comments w/ mentions: {len(comments_with_mentions)} ({comments_with_mentions_pct:.2f}%)")
print(f"Comments w/ digits: {len(comments_with_digits)} ({comments_with_digits_pct:.2f}%)")
print(f"Comments w/ punctuation: {len(comments_with_punctuation)}\n")

print(f"Max comment length: {max(comment_lengths) if comment_lengths else 0}")
print(f"Average comment length: {sum(comment_lengths)/len(comment_lengths):.2f}" if comment_lengths else "Avg. comment length: 0")

BASIC DATA EXPLORATION
Total comments: 46829
Unique videos: 109
Unique channels: 73
Unique authors: 35178

Short comments (<8 chars): 228
Comments w/ URLs: 15 (0.03%)
Comments w/ timestamps: 1461 (3.12%)
Comments w/ mentions: 3724 (7.95%)
Comments w/ digits: 6891 (14.72%)
Comments w/ punctuation: 40068

Max comment length: 9817
Average comment length: 97.74


In [15]:
with open(MET_GALA_ENTITIES_PATH, "r", encoding="utf-8") as f:
    met_gala_entities = json.load(f)

total_entities = len(met_gala_entities['entities'])
celebs = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'celebrity']
brands = [entity for entity in met_gala_entities['entities'].values() if entity['type'] == 'designer_brand']

print(f"Total entities: {total_entities}")
print(f"Total celebrities: {len(celebs)}")
print(f"Total brands: {len(brands)}")

Total entities: 481
Total celebrities: 361
Total brands: 120


In [16]:
# Prepare exact entity aliases after light cleaning is available
def build_entity_aliases(entities):
    entity_aliases = []
    for entity in entities:
        aliases = set()
        for alias in entity.get("aliases", []) + [entity["name"]]:
            clean_alias = clean_for_entity_matching(alias)
            if clean_alias:
                aliases.add(clean_alias)
        entity_aliases.append({"name": entity["name"], "aliases": sorted(aliases)})
    return entity_aliases

def find_entities(text, entity_aliases):
    clean_text = clean_for_entity_matching(text)
    padded_text = f" {clean_text} "
    found = []
    for entity in entity_aliases:
        for alias in entity["aliases"]:
            if f" {alias} " in padded_text:
                found.append(entity["name"])
                break
    return found

# Create exact alias matching for celeb and brand names
celeb_aliases = build_entity_aliases(celebs)
brand_aliases = build_entity_aliases(brands)


In [17]:
# Match celebrity and brand aliases in each comment
matched_comments = []
celeb_counter = Counter()
brand_counter = Counter()

for comment in comments_flattened:
    text_for_matching = comment.get("comment_text_entity", "")

    # Find entities in comment
    found_celebs = find_entities(text_for_matching, celeb_aliases)
    found_brands = find_entities(text_for_matching, brand_aliases)

    # Scoring on each counter
    for celeb in found_celebs:
        celeb_counter[celeb] += 1
    for brand in found_brands:
        brand_counter[brand] += 1

    comment["celebs"] = found_celebs
    comment["brands"] = found_brands

    # Append to matched comments
    matched_comments.append({
        **comment,
        "comment_id": comment.get("comment_id"),
        "video_id": comment.get("video_id"),
        "video_title": comment.get("video_title"),
        "comment_text": comment.get("comment_text", ""),
        "comment_text_entity": text_for_matching,
        "celebs": found_celebs,
        "brands": found_brands,
    })


In [18]:
comments_with_celebs = [row for row in matched_comments if row["celebs"]]
comments_with_brands = [row for row in matched_comments if row["brands"]]
# Which comments have both brand and celeb mentions 
# Expect this to be smaller
comments_with_both = [row for row in matched_comments if row["celebs"] and row["brands"]]

comments_with_celebs_pct = 100 * len(comments_with_celebs) / len(comments_flattened)
comments_with_brands_pct = 100 * len(comments_with_brands) / len(comments_flattened)
comments_with_both_pct = 100 * len(comments_with_both) / len(comments_flattened)

print("ENTITY MATCH COVERAGE")
print("=" * 80)
print(f"Total comments: {len(comments_flattened)}")
print(f"Comments with celebrity mentions: {len(comments_with_celebs)} ({comments_with_celebs_pct:.2f}%)")
print(f"Comments with brand mentions: {len(comments_with_brands)} ({comments_with_brands_pct:.2f}%)")
print(f"Comments with both celebrity and brand mentions: {len(comments_with_both)} ({comments_with_both_pct:.2f}%)")


ENTITY MATCH COVERAGE
Total comments: 46829
Comments with celebrity mentions: 10310 (22.02%)
Comments with brand mentions: 1003 (2.14%)
Comments with both celebrity and brand mentions: 351 (0.75%)


In [19]:
# Entities that are being found often enough to work with
matched_celeb_count = len(celeb_counter)
matched_brand_count = len(brand_counter)
matched_celeb_count_pct = 100 * matched_celeb_count / len(celebs)
matched_brand_count_pct = 100 * matched_brand_count / len(brands)

print("ENTITY FREQUENCY CHECK")
print("=" * 80)
print(f"Matched celebrities: {matched_celeb_count}/{len(celebs)} ({matched_celeb_count_pct:.2f}%)")
print(f"Matched brands: {matched_brand_count}/{len(brands)} ({matched_brand_count_pct:.2f}%)")


ENTITY FREQUENCY CHECK
Matched celebrities: 215/361 (59.56%)
Matched brands: 66/120 (55.00%)


In [20]:

print("\nTOP 20 CELEBRITIES")
print("=" * 80)
for celeb, count in celeb_counter.most_common(20):
    print(f"{celeb}: {count}")



TOP 20 CELEBRITIES
Beyonce: 1334
Jisoo: 1295
LISA: 1045
Rose: 939
Rihanna: 745
Madonna: 435
JENNIE: 411
Emma Chamberlain: 373
Heidi Klum: 333
Cardi B: 329
Anne Hathaway: 290
Kylie Jenner: 240
Katy Perry: 230
Bad Bunny: 223
Blake Lively: 210
Karan Johar: 206
Sabrina Carpenter: 199
Tyla: 183
Sam Smith: 182
Ningning: 163


In [21]:

print("\nTOP 20 BRANDS")
print("=" * 80)
for brand, count in brand_counter.most_common(20):
    print(f"{brand}: {count}")


TOP 20 BRANDS
Saint Laurent: 259
Robert Wun: 160
Dior: 87
Mugler: 77
Chanel: 69
Hugo Boss: 51
Balenciaga: 43
Schiaparelli: 40
Prada: 32
Zara: 24
Skims: 20
Zac Posen: 17
Gap Studio: 16
Valentino: 15
Chloe: 12
Tom Ford: 10
Christian Siriano: 9
Michael Kors: 9
Vivienne Westwood: 9
Bulgari: 9


In [24]:
# Store preprocessing variants on each filtered English comment
for comment in comments_flattened:
    original_text = comment.get("comment_text", "")
    topic_tokens = clean_for_topic_tokens(original_text)
    topic_stemmed_tokens = clean_for_topic_stemmed_tokens(original_text)

    # Used for VADER and BERT sentiment analysis
    comment["comment_text_sentiment_analysis"] = sentiment_analysis_clean(original_text)

    # Used for topic modelling
    comment["comment_text_topic"] = " ".join(topic_tokens)
    comment["comment_tokens_topic"] = topic_tokens
    comment["comment_text_topic_stemmed"] = " ".join(topic_stemmed_tokens)
    comment["comment_tokens_topic_stemmed"] = topic_stemmed_tokens
    comment["topic_token_count"] = len(topic_tokens)

print(f"Stored preprocessing variants for {len(comments_flattened)} English comments")


Stored preprocessing variants for 46829 English comments


In [25]:
# Demonstration of processing for random sample and report
RANDOM_TOTAL_EXAMPLES = 10
purify_random_samples = random.sample(english_comments, RANDOM_TOTAL_EXAMPLES)
for idx, text in enumerate(purify_random_samples):
    history = purify_text(text['comment_text'], True).items()
    print(f"\n[{idx+1}/{RANDOM_TOTAL_EXAMPLES}] {text['comment_text'][:100]}")
    for step, value in history:
        print(f"  [{step}] {value}")


[1/10] OMG Jisoo is so prettyyyy
  [remove_html_entities] OMG Jisoo is so prettyyyy
  [remove_urls] OMG Jisoo is so prettyyyy
  [remove_timestamps] OMG Jisoo is so prettyyyy
  [remove_mentions] OMG Jisoo is so prettyyyy
  [lowercase_and_strip] omg jisoo is so prettyyyy
  [normalise_whitespace] omg jisoo is so prettyyyy
  [remove_unicode] omg jisoo is so prettyyyy
  [remove_digits] omg jisoo is so prettyyyy
  [strip_punctuation] omg jisoo is so prettyyyy
  [normalise_whitespace_after_punctuation] omg jisoo is so prettyyyy
  [tokenize] ['omg', 'jisoo', 'is', 'so', 'prettyyy']
  [remove_stopwords] ['omg', 'jisoo', 'prettyyy']
  [stem_tokens] ['omg', 'jisoo', 'prettyyi']

[2/10] My faaaavorite thing is the blending up of masculine and feminine.  Just GORGE
  [remove_html_entities] My faaaavorite thing is the blending up of masculine and feminine.  Just GORGE
  [remove_urls] My faaaavorite thing is the blending up of masculine and feminine.  Just GORGE
  [remove_timestamps] My faaaavorite 

In [26]:

processed_video_data = {
    "comments": comments_flattened,
    "entity_counts": {
        "celebs": dict(celeb_counter),
        "brands": dict(brand_counter),
    },
}

with open(PROCESSED_VIDEO_DATA_PATH, "w", encoding="utf-8") as f:
    json.dump(processed_video_data, f, ensure_ascii=False, indent=2)

print(f"Saved processed data to {PROCESSED_VIDEO_DATA_PATH}")


Saved processed data to /Users/cooper/Documents/GitHub/Network-Analysis-Project#/data/video_data_processed.json


>